**Generating randomized data for created tables**.

In [0]:
import pandas as pd
import random
from datetime import datetime, timedelta

random.seed(42)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [0]:
spark.sql("SHOW TABLES").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|      chatbot_intent|      false|
| default|chatbot_interactions|      false|
| default|   customer_contacts|      false|
| default|           customers|      false|
| default|    escalation_tiers|      false|
| default|      issue_category|      false|
| default|   issue_subcategory|      false|
| default|            voc_tags|      false|
+--------+--------------------+-----------+



In [0]:
spark.sql("SELECT * FROM customers").show()

+-----------+----------------+-----------------+-----------+--------------+
|customer_id|customer_segment|subscription_tier|signup_date|account_status|
+-----------+----------------+-----------------+-----------+--------------+
+-----------+----------------+-----------------+-----------+--------------+



In [0]:
import pandas as pd
import random
from datetime import datetime, timedelta

customers = []

for i in range(1, 21):

    segment = random.choice([
        "New",
        "Existing",
        "At Risk"
    ])

    tier = random.choice([
        "Basic",
        "Standard",
        "Premium"
    ])

    status = random.choice([
        "Active",
        "Inactive"
    ])

    signup_date = datetime(2024, 1, 1) + timedelta(
        days=random.randint(0, 730)
    )

    customers.append([
        i,
        segment,
        tier,
        signup_date.date(),
        status
    ])

customers_df = pd.DataFrame(
    customers,
    columns=[
        "customer_id",
        "customer_segment",
        "subscription_tier",
        "signup_date",
        "account_status"
    ]
)

customers_df.head()

,customer_id,customer_segment,subscription_tier,signup_date,account_status
0,1,At Risk,Basic,2025-09-27,Active
1,2,New,Premium,2024-09-18,Inactive
2,3,Existing,Standard,2025-03-08,Active
3,4,Existing,Basic,2024-06-12,Inactive
4,5,New,Premium,2024-06-08,Inactive


In [0]:
customers_spark = spark.createDataFrame(customers_df)

customers_spark.show()

+-----------+----------------+-----------------+-----------+--------------+
|customer_id|customer_segment|subscription_tier|signup_date|account_status|
+-----------+----------------+-----------------+-----------+--------------+
|          1|         At Risk|            Basic| 2025-09-27|        Active|
|          2|             New|          Premium| 2024-09-18|      Inactive|
|          3|        Existing|         Standard| 2025-03-08|        Active|
|          4|        Existing|            Basic| 2024-06-12|      Inactive|
|          5|             New|          Premium| 2024-06-08|      Inactive|
|          6|         At Risk|         Standard| 2025-04-14|        Active|
|          7|             New|            Basic| 2025-02-03|      Inactive|
|          8|         At Risk|         Standard| 2025-01-21|        Active|
|          9|        Existing|         Standard| 2024-07-13|      Inactive|
|         10|             New|          Premium| 2024-10-10|        Active|
|         11

In [0]:
customers_spark.write.mode("append").saveAsTable("customers")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6461330411565776>, line 1
----> 1 customers_spark.write.mode("append").saveAsTable("customers")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_id
   1537 self._set_command_in_plan(req.plan, command)
-> 1538 

In [0]:
print(customers_df.columns.tolist())

['customer_id', 'customer_segment', 'subscription_tier', 'signup_date', 'account_status']


In [0]:
customers_spark.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- subscription_tier: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- account_status: string (nullable = true)



In [0]:
spark.sql("DESCRIBE customers").show(truncate=False)

+-----------------+-----------+-------+
|col_name         |data_type  |comment|
+-----------------+-----------+-------+
|customer_id      |int        |NULL   |
|customer_segment |varchar(20)|NULL   |
|subscription_tier|varchar(20)|NULL   |
|signup_date      |date       |NULL   |
|account_status   |varchar(10)|NULL   |
+-----------------+-----------+-------+



In [0]:
from pyspark.sql.functions import col

customers_spark_clean = customers_spark.select(
    col("customer_id").cast("int").alias("customer_id"),
    col("customer_segment").cast("string").alias("customer_segment"),
    col("subscription_tier").cast("string").alias("subscription_tier"),
    col("signup_date").cast("date").alias("signup_date"),
    col("account_status").cast("string").alias("account_status")
)

customers_spark_clean.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- subscription_tier: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- account_status: string (nullable = true)



In [0]:
customers_spark_clean.write.mode("append").insertInto("customers")

In [0]:
spark.sql("SELECT * FROM customers").show()

+-----------+----------------+-----------------+-----------+--------------+
|customer_id|customer_segment|subscription_tier|signup_date|account_status|
+-----------+----------------+-----------------+-----------+--------------+
|          1|         At Risk|            Basic| 2025-09-27|        Active|
|          2|             New|          Premium| 2024-09-18|      Inactive|
|          3|        Existing|         Standard| 2025-03-08|        Active|
|          4|        Existing|            Basic| 2024-06-12|      Inactive|
|          5|             New|          Premium| 2024-06-08|      Inactive|
|          6|         At Risk|         Standard| 2025-04-14|        Active|
|          7|             New|            Basic| 2025-02-03|      Inactive|
|          8|         At Risk|         Standard| 2025-01-21|        Active|
|          9|        Existing|         Standard| 2024-07-13|      Inactive|
|         10|             New|          Premium| 2024-10-10|        Active|
|         11

In [0]:
issue_category_data = [
    (1, "Card Issues"),
    (2, "Direct Deposit"),
    (3, "E-Transfer"),
    (4, "Cashback Rewards"),
    (5, "App Login"),
    (6, "Subscription Billing"),
    (7, "Fraud Security")
]

issue_category_df = spark.createDataFrame(
    issue_category_data,
    ["issue_category_id", "issue_category_name"]
)

issue_category_df.write.mode("append").insertInto("issue_category")

In [0]:
spark.sql("SELECT * FROM issue_category").show()

+-----------------+--------------------+
|issue_category_id| issue_category_name|
+-----------------+--------------------+
|                1|         Card Issues|
|                2|      Direct Deposit|
|                3|          E-Transfer|
|                4|    Cashback Rewards|
|                5|           App Login|
|                6|Subscription Billing|
|                7|      Fraud Security|
+-----------------+--------------------+



In [0]:
escalation_tiers_data = [
    (1, "Bot Only", "Resolved by chatbot without agent support"),
    (2, "Tier 1", "Handled by frontline support"),
    (3, "Tier 2", "Escalated to specialist support"),
    (4, "Specialist Team", "Escalated to specialized internal team")
]

escalation_tiers_df = spark.createDataFrame(
    escalation_tiers_data,
    ["escalation_tier_id", "tier_name", "tier_description"]
)

escalation_tiers_df.write.mode("append").insertInto("escalation_tiers")

In [0]:
spark.sql("SELECT * FROM escalation_tiers").show()

+------------------+---------------+--------------------+
|escalation_tier_id|      tier_name|    tier_description|
+------------------+---------------+--------------------+
|                 1|       Bot Only|Resolved by chatb...|
|                 2|         Tier 1|Handled by frontl...|
|                 3|         Tier 2|Escalated to spec...|
|                 4|Specialist Team|Escalated to spec...|
+------------------+---------------+--------------------+



In [0]:
issue_subcategory_data = [

    (1, "Lost Card", 1),
    (2, "Stolen Card", 1),
    (3, "Failed ATM Transaction", 1),

    (4, "Delayed Deposit", 2),
    (5, "Failed Deposit", 2),

    (6, "Failed E-Transfer", 3),
    (7, "Stuck E-Transfer", 3),
    (8, "Fraudulent E-Transfer", 3),

    (9, "Missing Cashback", 4),
    (10, "Referral Bonus", 4),

    (11, "Password Reset", 5),
    (12, "App Crash", 5),

    (13, "Upgrade Subscription", 6),
    (14, "Billing Error", 6),

    (15, "Account Takeover", 7),
    (16, "Identity Theft", 7)

]

issue_subcategory_df = spark.createDataFrame(
    issue_subcategory_data,
    ["issue_subcategory_id",
     "issue_subcategory_name",
     "issue_category_id"]
)

issue_subcategory_df.write.mode("append").insertInto("issue_subcategory")

In [0]:
spark.sql("SELECT * FROM issue_subcategory").show()

+--------------------+----------------------+-----------------+
|issue_subcategory_id|issue_subcategory_name|issue_category_id|
+--------------------+----------------------+-----------------+
|                   1|             Lost Card|                1|
|                   2|           Stolen Card|                1|
|                   3|  Failed ATM Transa...|                1|
|                   4|       Delayed Deposit|                2|
|                   5|        Failed Deposit|                2|
|                   6|     Failed E-Transfer|                3|
|                   7|      Stuck E-Transfer|                3|
|                   8|  Fraudulent E-Tran...|                3|
|                   9|      Missing Cashback|                4|
|                  10|        Referral Bonus|                4|
|                  11|        Password Reset|                5|
|                  12|             App Crash|                5|
|                  13|  Upgrade Subscrip

In [0]:
chatbot_intent_data = [

    (1, "Password Reset", "Authentication"),
    (2, "Card Activation", "Cards"),
    (3, "Card Status", "Cards"),
    (4, "Direct Deposit Issue", "Deposits"),
    (5, "E-Transfer Issue", "Payments"),
    (6, "Rewards Inquiry", "Rewards"),
    (7, "Subscription Inquiry", "Billing"),
    (8, "Fraud Investigation", "Security")

]

chatbot_intent_df = spark.createDataFrame(
    chatbot_intent_data,
    ["intent_id",
     "intent_name",
     "intent_category"]
)

chatbot_intent_df.write.mode("append").insertInto("chatbot_intent")

In [0]:
spark.sql("SELECT * FROM chatbot_intent").show()

+---------+--------------------+---------------+
|intent_id|         intent_name|intent_category|
+---------+--------------------+---------------+
|        1|      Password Reset| Authentication|
|        2|     Card Activation|          Cards|
|        3|         Card Status|          Cards|
|        4|Direct Deposit Issue|       Deposits|
|        5|    E-Transfer Issue|       Payments|
|        6|     Rewards Inquiry|        Rewards|
|        7|Subscription Inquiry|        Billing|
|        8| Fraud Investigation|       Security|
+---------+--------------------+---------------+



In [0]:
spark.sql("""
SELECT *
FROM customers
""").show()

+-----------+----------------+-----------------+-----------+--------------+
|customer_id|customer_segment|subscription_tier|signup_date|account_status|
+-----------+----------------+-----------------+-----------+--------------+
|          1|         At Risk|            Basic| 2025-09-27|        Active|
|          2|             New|          Premium| 2024-09-18|      Inactive|
|          3|        Existing|         Standard| 2025-03-08|        Active|
|          4|        Existing|            Basic| 2024-06-12|      Inactive|
|          5|             New|          Premium| 2024-06-08|      Inactive|
|          6|         At Risk|         Standard| 2025-04-14|        Active|
|          7|             New|            Basic| 2025-02-03|      Inactive|
|          8|         At Risk|         Standard| 2025-01-21|        Active|
|          9|        Existing|         Standard| 2024-07-13|      Inactive|
|         10|             New|          Premium| 2024-10-10|        Active|
|         11

In [0]:
import pandas as pd
import random
from datetime import datetime, timedelta
from pyspark.sql.functions import col

random.seed(42)

contacts = []

# Weighted issue categories to make the analysis interesting
issue_weights = [
    (1, [1, 2, 3], 20),   # Card Issues
    (2, [4, 5], 25),      # Direct Deposit
    (3, [6, 7, 8], 18),   # E-Transfer
    (4, [9, 10], 10),     # Cashback Rewards
    (5, [11, 12], 12),    # App Login
    (6, [13, 14], 7),     # Subscription Billing
    (7, [15, 16], 8)      # Fraud Security
]

weighted_categories = []
for category_id, subcategories, weight in issue_weights:
    weighted_categories.extend([(category_id, subcategories)] * weight)

channels = ["Chatbot", "Email", "Phone", "In-App Chat"]

for contact_id in range(1, 101):
    customer_id = random.randint(1, 20)
    issue_category_id, subcategories = random.choice(weighted_categories)
    issue_subcategory_id = random.choice(subcategories)
    contact_date = datetime(2026, 1, 1) + timedelta(days=random.randint(0, 150))
    channel = random.choice(channels)

    # Escalation logic
    if issue_category_id == 7:      # Fraud/security
        escalation_tier_id = random.choices([2, 3, 4], weights=[30, 50, 20])[0]
    elif channel == "Chatbot":
        escalation_tier_id = random.choices([1, 2, 3], weights=[55, 35, 10])[0]
    else:
        escalation_tier_id = random.choices([2, 3, 4], weights=[70, 25, 5])[0]

    resolved_flag = random.choices([True, False], weights=[82, 18])[0]

    resolution_time_minutes = random.randint(5, 240)

    # Lower CSAT for unresolved/escalated contacts
    if not resolved_flag or escalation_tier_id in [3, 4]:
        csat_score = round(random.uniform(45, 75), 2)
    else:
        csat_score = round(random.uniform(70, 98), 2)

    # Higher cost for higher escalation
    cost_map = {
        1: random.uniform(0.50, 2.50),
        2: random.uniform(3.00, 7.00),
        3: random.uniform(8.00, 15.00),
        4: random.uniform(16.00, 30.00)
    }

    cost_per_contact = round(cost_map[escalation_tier_id], 2)

    contacts.append([
        contact_id,
        customer_id,
        contact_date.date(),
        channel,
        issue_category_id,
        issue_subcategory_id,
        escalation_tier_id,
        resolved_flag,
        resolution_time_minutes,
        csat_score,
        cost_per_contact
    ])

customer_contacts_df = pd.DataFrame(
    contacts,
    columns=[
        "contact_id",
        "customer_id",
        "contact_date",
        "channel",
        "issue_category_id",
        "issue_subcategory_id",
        "escalation_tier_id",
        "resolved_flag",
        "resolution_time_minutes",
        "csat_score",
        "cost_per_contact"
    ]
)

customer_contacts_spark = spark.createDataFrame(customer_contacts_df)

customer_contacts_spark_clean = customer_contacts_spark.select(
    col("contact_id").cast("int"),
    col("customer_id").cast("int"),
    col("contact_date").cast("date"),
    col("channel").cast("string"),
    col("issue_category_id").cast("int"),
    col("issue_subcategory_id").cast("int"),
    col("escalation_tier_id").cast("int"),
    col("resolved_flag").cast("boolean"),
    col("resolution_time_minutes").cast("int"),
    col("csat_score").cast("decimal(5,2)"),
    col("cost_per_contact").cast("decimal(5,2)")
)

customer_contacts_spark_clean.write.mode("append").insertInto("customer_contacts")

In [0]:
spark.sql("SELECT * FROM customer_contacts LIMIT 10").show()

+----------+-----------+------------+-----------+-----------------+--------------------+------------------+-------------+-----------------------+----------+----------------+
|contact_id|customer_id|contact_date|    channel|issue_category_id|issue_subcategory_id|escalation_tier_id|resolved_flag|resolution_time_minutes|csat_score|cost_per_contact|
+----------+-----------+------------+-----------+-----------------+--------------------+------------------+-------------+-----------------------+----------+----------------+
|         1|          4|  2026-03-12|      Email|                1|                   3|                 2|         true|                    178|     90.74|            5.36|
|         2|          8|  2026-05-24|      Email|                4|                   9|                 3|         true|                    112|     51.61|            8.05|
|         3|         14|  2026-02-09|      Email|                2|                   5|                 4|         true|         

In [0]:
spark.sql("""
SELECT
    issue_category_id,
    COUNT(*) AS contacts
FROM customer_contacts
GROUP BY issue_category_id
ORDER BY contacts DESC
""").show()

+-----------------+--------+
|issue_category_id|contacts|
+-----------------+--------+
|                2|      19|
|                1|      18|
|                3|      17|
|                4|      16|
|                6|      12|
|                5|      12|
|                7|       6|
+-----------------+--------+



In [0]:
spark.sql("""
SELECT COUNT(*)
FROM customer_contacts
""").show()

+--------+
|COUNT(*)|
+--------+
|     100|
+--------+



In [0]:
import pandas as pd
import random
from pyspark.sql.functions import col

random.seed(42)

chatbot_interactions = []

for chatbot_id in range(1, 71):

    contact_id = chatbot_id

    customer_id = random.randint(1, 20)

    intent_id = random.choices(
        [1,2,3,4,5,6,7,8],
        weights=[20,15,10,20,15,10,5,5]
    )[0]

    bot_confidence_score = round(
        random.uniform(0.60, 0.99),
        2
    )

    # Intent-based behavior
    if intent_id == 1:  # Password Reset
        bot_resolved_flag = random.choices(
            [True, False],
            weights=[90,10]
        )[0]

    elif intent_id == 8:  # Fraud Investigation
        bot_resolved_flag = random.choices(
            [True, False],
            weights=[15,85]
        )[0]

    else:
        bot_resolved_flag = random.choices(
            [True, False],
            weights=[65,35]
        )[0]

    handoff_to_agent_flag = not bot_resolved_flag

    fallback_flag = random.choices(
        [True, False],
        weights=[15,85]
    )[0]

    chatbot_interactions.append([
        chatbot_id,
        customer_id,
        contact_id,
        intent_id,
        bot_confidence_score,
        bot_resolved_flag,
        handoff_to_agent_flag,
        fallback_flag
    ])

chatbot_df = pd.DataFrame(
    chatbot_interactions,
    columns=[
        "chatbot_id",
        "customer_id",
        "contact_id",
        "intent_id",
        "bot_confidence_score",
        "bot_resolved_flag",
        "handoff_to_agent_flag",
        "fallback_flag"
    ]
)

In [0]:
chatbot_spark = spark.createDataFrame(chatbot_df)

chatbot_spark_clean = chatbot_spark.select(
    col("chatbot_id").cast("int"),
    col("customer_id").cast("int"),
    col("contact_id").cast("int"),
    col("intent_id").cast("int"),
    col("bot_confidence_score").cast("decimal(5,2)"),
    col("bot_resolved_flag").cast("boolean"),
    col("handoff_to_agent_flag").cast("boolean"),
    col("fallback_flag").cast("boolean")
)

In [0]:
chatbot_spark_clean.printSchema()

root
 |-- chatbot_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- contact_id: integer (nullable = true)
 |-- intent_id: integer (nullable = true)
 |-- bot_confidence_score: decimal(5,2) (nullable = true)
 |-- bot_resolved_flag: boolean (nullable = true)
 |-- handoff_to_agent_flag: boolean (nullable = true)
 |-- fallback_flag: boolean (nullable = true)



In [0]:
from pyspark.sql.functions import col, lit

chatbot_spark_clean = chatbot_spark_clean.withColumn(
    "interaction_date",
    lit("2026-03-01").cast("date")
)

chatbot_spark_clean = chatbot_spark_clean.select(
    col("chatbot_id").cast("int"),
    col("customer_id").cast("int"),
    col("contact_id").cast("int"),
    col("interaction_date").cast("date"),
    col("intent_id").cast("int"),
    col("bot_confidence_score").cast("decimal(5,2)"),
    col("bot_resolved_flag").cast("boolean"),
    col("handoff_to_agent_flag").cast("boolean"),
    col("fallback_flag").cast("boolean")
)

chatbot_spark_clean.write.mode("append").insertInto("chatbot_interactions")

In [0]:
spark.sql("SELECT * FROM chatbot_interactions LIMIT 10").show()
spark.sql("SELECT COUNT(*) FROM chatbot_interactions").show()

+----------+-----------+----------+----------------+---------+--------------------+-----------------+---------------------+-------------+
|chatbot_id|customer_id|contact_id|interaction_date|intent_id|bot_confidence_score|bot_resolved_flag|handoff_to_agent_flag|fallback_flag|
+----------+-----------+----------+----------------+---------+--------------------+-----------------+---------------------+-------------+
|         1|          4|         1|      2026-03-01|        1|                0.71|             true|                false|        false|
|         2|         18|         2|      2026-03-01|        1|                0.76|             true|                false|        false|
|         3|         17|         3|      2026-03-01|        4|                0.82|            false|                 true|        false|
|         4|         14|         4|      2026-03-01|        2|                0.83|            false|                 true|         true|
|         5|          6|         5

In [0]:
import pandas as pd
import random
from pyspark.sql.functions import col

random.seed(42)

voc_rows = []

sentiment_options = ["Positive", "Neutral", "Negative"]
pain_points = [
    "Delayed Resolution",
    "Confusing App Experience",
    "Long Wait Time",
    "Poor Communication",
    "Missing Feature",
    "Unclear Policy",
    "Technical Issue"
]
root_causes = [
    "System Bug",
    "Policy Restriction",
    "User Error",
    "Process Gap",
    "Third-Party Dependency",
    "Knowledge Gap",
    "Fraud Review Required"
]

for voc_tag_id in range(1, 101):
    contact_id = voc_tag_id

    contact = spark.sql(f"""
        SELECT issue_category_id, resolved_flag
        FROM customer_contacts
        WHERE contact_id = {contact_id}
    """).collect()[0]

    issue_category_id = contact["issue_category_id"]
    resolved_flag = contact["resolved_flag"]

    if issue_category_id == 7:  # Fraud/security
        sentiment_label = random.choices(sentiment_options, weights=[10, 25, 65])[0]
        root_cause = random.choice(["Fraud Review Required", "Policy Restriction", "Process Gap"])
    elif resolved_flag == False:
        sentiment_label = random.choices(sentiment_options, weights=[10, 30, 60])[0]
        root_cause = random.choice(["System Bug", "Process Gap", "Third-Party Dependency"])
    else:
        sentiment_label = random.choices(sentiment_options, weights=[35, 45, 20])[0]
        root_cause = random.choice(root_causes)

    sentiment_score = {
        "Positive": round(random.uniform(0.30, 1.00), 2),
        "Neutral": round(random.uniform(-0.20, 0.20), 2),
        "Negative": round(random.uniform(-1.00, -0.30), 2)
    }[sentiment_label]

    complaint_flag = sentiment_label == "Negative"

    repeat_contact_flag = random.choices(
        [True, False],
        weights=[35, 65] if complaint_flag else [15, 85]
    )[0]

    pain_point = random.choice(pain_points)

    voc_rows.append([
        voc_tag_id,
        contact_id,
        sentiment_label,
        sentiment_score,
        pain_point,
        complaint_flag,
        repeat_contact_flag,
        root_cause
    ])

voc_df = pd.DataFrame(
    voc_rows,
    columns=[
        "voc_tag_id",
        "contact_id",
        "sentiment_label",
        "sentiment_score",
        "pain_point",
        "complaint_flag",
        "repeat_contact_flag",
        "root_cause"
    ]
)

voc_spark = spark.createDataFrame(voc_df)

voc_spark_clean = voc_spark.select(
    col("voc_tag_id").cast("int"),
    col("contact_id").cast("int"),
    col("sentiment_label").cast("string"),
    col("sentiment_score").cast("decimal(3,2)"),
    col("pain_point").cast("string"),
    col("complaint_flag").cast("boolean"),
    col("repeat_contact_flag").cast("boolean"),
    col("root_cause").cast("string")
)

voc_spark_clean.write.mode("append").insertInto("voc_tags")

In [0]:
spark.sql("SELECT COUNT(*) FROM voc_tags").show()
spark.sql("SELECT * FROM voc_tags LIMIT 10").show()

+--------+
|COUNT(*)|
+--------+
|     100|
+--------+

+----------+----------+---------------+---------------+--------------------+--------------+-------------------+--------------------+
|voc_tag_id|contact_id|sentiment_label|sentiment_score|          pain_point|complaint_flag|repeat_contact_flag|          root_cause|
+----------+----------+---------------+---------------+--------------------+--------------+-------------------+--------------------+
|         1|         1|       Positive|           0.32|Confusing App Exp...|         false|              false|Third-Party Depen...|
|         2|         2|        Neutral|          -0.20|      Long Wait Time|         false|              false|          User Error|
|         3|         3|       Positive|           0.97|     Technical Issue|         false|               true|  Policy Restriction|
|         4|         4|        Neutral|           0.09|  Poor Communication|         false|              false|         Process Gap|
|         5| 